# What is a representation, and why does it matter for similarity joins?

*Article 3 of 3 in the Similarity Joins series*

## Looking back

In Articles 1 and 2 we looked at similarity joins on entity representations. While the term representation was used, it was not really discussed what a representation is. This article dives beneath the surface to give a more in depth look at representations.

- **Article 1** introduced similarity joins with three cardinality shapes: kNN-join, k-closest pair, ε-join.
- **Article 2** added multiple representations per entity (an **attribute rep** from text and a **relational rep** from many-to-many structure), cross-entity joins via shared latent spaces, and group aggregation via the function `rep_agg`.

Representations such as embeddings sit underneath nearly every modern AI system: RAG, recommenders, semantic search, classification, agent memory. They're treated as opaque vectors that 'just work'. 

## Learning outcomes

This article gives a working framework on similarity joins and representations: what we can do with them, what a representation actually is (from an information-theoretic view), what makes one useful (task-relative utility), and how design patterns turn into bets on the task distribution.

In this article we will

- delve into what a representation is and what makes one good
- go over considerations in representation design
- understand related approaches to similarity joins


## 1. What else can we do with representations?

In our view, representations as columns and similarity as a first-class operator give databases a class of capabilities they didn't have before: prediction at query time, population-level analytics over similarity, cross-entity reasoning through shared spaces, and similarity-aware structured output that downstream consumers (LLMs included) can reason over. Vector-DB-style retrieval is one corner of this surface. The table below lists a sample of the questions this opens up, not an exhaustive map.

| Question | Task | Operation |
|---|---|---|
| What products will this user like? | Recommendation / regression | kNN-join: user rep → product reps |
| Who will likely buy this product? | Threshold prediction | ε-join: user × product rep with `gte=ε` |
| Which job fits this applicant? | Recommendation | kNN-join: applicant → job reps |
| Which jobs have a thin talent pool? | Match-volume analytics | ε-join + `GROUP BY job` + `COUNT` |
| What treatment fits this patient? | Classification | kNN-join: patient → labelled patients, vote labels |
| Which research papers are relevant to this case? | Search | `vec_filter` on paper rep |
| Is this transaction anomalous? | Anomaly detection | Distance to k-th neighbour on transaction rep |
| Are these two accounts the same person? | Dedup / entity resolution | Self ε-join, `gte=0.9` |
| What's the typical profile of an engineer? | Prototype retrieval | `rep_agg` over engineering occupations |
| How to recommend for a new user with no purchase history? | Cold-start | Use attribute rep (profile text, demographics) or `rep_agg` over similar users' purchases |
| Discover customer segments without labels | Clustering | Self ε-join + connected components |

The questions cluster around a few kinds: per-row prediction (kNN-join + label aggregate), threshold-based population queries (ε-join + `GROUP BY`), centroid-based search (`rep_agg` + `vec_filter`), and so on. The rest of this article explains why these work: what a representation is, what makes one useful, and how to design one. If you only read this section, you'll still walk away with a recipe for the most common questions.

## 2. What is a representation?

Representations are often discussed vaguely. To help avoid that, we'll separate the following three concerns:

- **Definition** - what a representation actually *is*, structurally.
- **Utility** - what it's *good for*, given some task.
- **Design** - *how* it gets built, and the bets that go into building it.

So what is a representation? Well, there has been a lot of discussion on this, but I find that motivating it by information theory is the easiest to work with. Here we'll borrow from Shannon, Tishby, and Fisher and also decouple how to view data (what space to transform it to and what information to include) from how useful that view is (the utility of the view).

First let's imagine we have an entity `X` which has information `H`. We don't see `X` directly, but have access to other objects that tell us about it. A representation of `X` can be any object `Z` with positive mutual information `I(Z; X) > 0`, equivalently, knowing `Z` reduces uncertainty about `X`. From any representation we can derive new ones. We can get different views of the entity by transforming data that we have represented the entity with or selecting more data to represent it with. To transform it, we use an encoder, a function `f` which converts from one view to another. Applied to a representation `D`, an encoder produces a new representation `Z = f(D)` in the output view (or `Z ∼ p(Z | D, context)` for stochastic encoders). The information profile `I(Z; X | context)` measures how much information `Z` carries about `X`.

X could be any entity. Let's say it is the entity "fruit" and we have a row called "apple". This entity can have multiple attributes. For example, an attribute could be a single exemplar image of that apple. Then we choose a representation for this entity. One rep could just be the pixel values for the attribute, another could be the principal components of those pixel values. We could also have a list of vitamins in it and derive representations using a text encoder as well or even combine that info with the principal components. All of these will have different similarity values and all of them reduce uncertainty about X.

- **Entity**: A thing with identity. In database terms, typically a row; more abstractly, anything with a stable referent (a person, a product, an occupation).
- **Information**: What reduces uncertainty about an entity. Same information, many possible data forms.
- **Representation**: An object `Z` with positive mutual information with the entity `X` it represents (`I(Z; X) > 0`) - knowing `Z` reduces uncertainty about `X`. Usually stored as a typed column (a vector or other typed value), sitting as a point in some view. In machine learning, it is a set of **features** that describe an entity. Its information profile, `I(Z; X)`, gives us how much `Z` tells us about `X`. We can get different views by using an encoder to change the space the data is in, `f: V_in → V_out`.
- **Attribute rep**: A representation using or derived from one or more of an entity's own attributes (for example, a sentence-transformer encoding of a text column). 
- **Relational rep**: A representation derived from an entity's relationships to other entities (for example, an ALS factorisation of a many-to-many table - see Article 2).

Each transformation we use in encoding can only preserve or lose information about the entity. The data-processing inequality is the formal floor. For any deterministic encoder `Z = f(D)`: `I(Z; X) ≤ I(D; X)`. A text embedding doesn't know more about a document than the raw text does but gives us new ways to express similarity.

In "representation learning", we use machine learning techniques to learn the representations for entities with techniques such as word2vec or LLM-based embedding models. These convert one-hot encodings (a type of sparse vector) to vectors in a space that allows for similarity computation. This learns a bundle of features describing the entity.

This bundle-of-features framing aligns with how Goodfellow, Bengio, and Courville (2016, ch. 15) and Bengio-Courville-Vincent 2013 treat representations: features as components, the representation as the bundle they form. Both focus on what makes a representation useful rather than what it is.

Other rigorous formulations exist (Shannon's sufficient statistics, group-theoretic representations). These often entangle the definition with design or utility considerations.

## 3. Good representations through utility

Any transformation (even the identity) on the attributes of the entity will be a representation, but that doesn't necessarily mean the representation will be useful. Here we look at utility, which will help us with usefulness.

We can frame utility as the expected performance of a downstream system on a task:

`U(rep, T) = 𝔼[ score(f_T(rep(x)), y) ]` over `(x, y)` drawn from `T`.

`f_T` is the consumer, which could be anything: a classifier, a similarity query, a generator, and so on. `score` is whatever metric measures success. `T` is the task. For similarity-join workloads, `f_T` is the metric comparing two representations (cosine, Euclidean, KL, and so on), and the score measures whether that metric tracks the "ground-truth relationship".

A representation will often be needed for more than one task so in that case we can measure it based on its performance on multiple tasks. For example with the following formula,

`U_general(rep) = 𝔼[ U(rep, T) ]` over `T ∼ P(tasks)`.

Scoring well across a broad task distribution is what "general-purpose representation" means in practice. Under any non-degenerate task distribution no rep maximises `U` for every task simultaneously (Wolpert 1996, no-free-lunch). So we can say utility is ultimately dependent on the task and not an intrinsic property of a representation.

For example, the one-hot encodings mentioned above have low utility when it comes to similarity joins and thus provide little utility in our case because they only produce non-zero similarity when two vectors are identical. 

Text embeddings on the other hand map the one-hot encodings into a space that helps us understand the semantic similarity of two different text strings in that space.

## 4. Similarity functions

In Article 1, we strolled through similarity joins: **ε-join** for thresholds (Silva-Aref-Ali 2010, with Chaudhuri-Ganti-Kaushik 2006 and Gravano et al. 2001 as data-cleaning predecessors), **kNN-join** for per-outer top-K (Böhm-Krebs 2004), **k-closest pair join** for global top-N (Hjaltason-Samet 1998, Corral et al. 2000). The shapes differ in algebraic character and in the prediction pattern each enables.

kNN-join is bivalent: search and prediction sit under the same operator. Once representations are stored as columns, the database performs prediction at query time. This turns the database into a predictive model. The model is non-parametric and defined by the data and the similarity metric.

### Join Type

**ε-join.** It returns everything that passes the threshold. It is symmetric, has its output cardinality determined by data, and is an extension of the equi-join. It also composes cleanly with the rest of relational algebra. It is represented by the formula `A ⋈_ε B = { (a, b) | d(a.rep, b.rep) ≤ ε }`, where d is a distance function. If it is a similarity function, then the inequality will be replaced with `≥`. It can be used for similarity graph construction, finding dissimilar items, and population analytics such as identifying generalists and specialists.

**kNN-join** has its cardinality fixed. This join is appropriate when you need a match for every row and enables us to do prediction, retrieval, and density estimation. It is represented by the formula `kNN(A, B, k) = ⋃_{a ∈ A} { (a, b) | b ∈ topK_d(a, B) }`, always returning `k × |A|` rows. As a prediction shape, it gives row-level prediction with a guaranteed sample size: recommend K items per user, classify by the K labelled neighbours, suggest K career paths per applicant. It is useful when downstream code expects fixed-shape output.

**k-closest pair join** (Silva-Aref-Ali 2010 call this the **k-Distance Join**, or **kD-Join**) is another retrieval-style join. It returns exactly the N pairs with smallest distance across the cross-product. It gives leaderboard-style outputs which are ranked-list results across the whole cross-product, not sliced per-outer. It is useful for duplicate detection, deduplication, and "best matches we have right now" surfaces.

The choice comes down to the question. Hybrid combinations compose with thresholds: `k=10, gte=0.5` is "up to 10 neighbours, but only if they clear 0.5".

Silva-Aref-Ali 2010 also formalises a fourth shape this article series does not exercise: **Join-Around** (or **A-Join**). For each row on the left it returns the single closest row on the right, but only if that row falls within a radius `r`. With `r=∞` it reduces to kNN-join with `k=1`; the radius cap is what makes it distinct. Silva singles it out because it composes naturally with their Similarity-Group-By-Around operator. It is the "assign each input to its nearest reference point" pattern, the join behind Voronoi-style partitioning and 1-NN classification.

### Metric

Beyond that, we need a metric (similarity, distance, divergence, etc.) on the representation space. The valid metrics depend on the geometry of the space:

- Dense float vectors (R^d): cosine similarity, Euclidean, inner product, Mahalanobis distance
- Bit vectors ({0,1}^d): Hamming distance
- Strings: Levenshtein distance
- Sets: Jaccard, Dice
- Distributions (simplex): JS divergence, Wasserstein distance

Switching metrics on the same column changes what "similar" means, often dramatically. Picking the metric is part of representation design, not a step that happens after.

Also, KL divergence can be a good choice for similarity, but since it is not symmetric you can get different pairs depending on the direction of the join.

## 5. Representing sets of data with aggregation

While above we discussed the representation of a single row, we can also use aggregation to get the representation of multiple rows since hte 
the aggregate of a set of representations is itself a representation. It has its own information profile (bounded above by the joint information of its members), its own utility profile (depends on what it gets used for), and its own design choice (i.e. which aggregation rule). For aggregation, the library provides `mean`, `sum`, `weighted_mean` for aggregation. Far more could be used like `max`, `min`, `geometric_mean`, or other `t_norms` and `s_norms` taken from fuzzy set theory.

With `weighted_mean` we can also do similarity weighted aggregation. This is the structure of kNN regression / Nadaraya-Watson kernel smoothing, attention (softmax of similarities, then weighted sum of values), and RAG-style evidence weighting. The weights themselves come from a similarity computation rather than from stored data.

We can also do column-wise aggregation to aggregate multiple scores together if multiple similarity calculations are computed. The same aggregation operators are currently available, but far more could be used like `max`, `min`, `geometric_mean`, or other `t_norms` and `s_norms` taken from fuzzy set theory.

## 6. Designing and selecting good representations

Since it is not always clear what representation we need up front for the database, it can be useful to follow good design heuristics that will produce representations that will work in a lot of cases. The representation can use a hand-programmed transformation like a Fourier Transform or PCA or it can also be learned. Bengio–Courville–Vincent 2013 listed several design patterns for representations. These are prescriptive, empirically-grounded heuristics betting that the task distribution will reward that property, but don't guarantee utility. 

Here are a few with the underlying engineering principle:
- **Smoothness** (locality/coherence): instances are similar to their neighbours. This aims to get local coherence in your representations by having similar positions output similar values.
- **Distributed** (compositionality): each concept is represented by a pattern across many features, and each feature participates in many concepts. Concepts compose from features, giving exponentially more representable states than one-hot encodings.
- **Disentangled** (cohesion/decoupling): each dimension has a "responsibility" for one concept rather than entangling many concepts across dimensions.
- **Invariant** (robustness): the rep is unchanged under specific transformations. For example, a translation-invariant rep produces the same output regardless of where the input is shifted.
- **Hierarchical** (abstraction/decomposition): the rep is organized into levels where lower level features compose into higher level ones.

In representation learning, the learning machine is designed to encourage such properties.

## 7. Related approaches

The framework here positions this library inside a wider field. Five neighbouring threads worth knowing:

- **Vector databases** (Pinecone, Weaviate, Milvus, Chroma, Qdrant): embedding-store-as-product, optimised for kNN retrieval at scale with managed ANN indexes.
- **Vector columns** (pgvector, DuckDB VSS): vector similarity operations added to existing relational engines. The layer this library builds on.
- **ANN libraries** (FAISS, ScaNN, Annoy, hnswlib): algorithms for similarity but not databases. Bring your own join logic.
- **LLM-driven semantic joins** (LOTUS and such): use an LLM as the join *predicate* - for each candidate pair, prompt the model to decide. A different layer of the stack: LLM-call-as-comparator rather than learned-rep-as-coordinate. Higher per-pair cost, more flexible per-question, doesn't compose with similarity geometry. Strong when the predicate genuinely needs reasoning ("is this paper a counterexample to that claim?") rather than geometric proximity.
- **Classical fuzzy joins** (Splink, Dedupe, ZinGG; Chaudhuri-Ganti-Kaushik 2006; Gravano et al. 2001): the data-integration lineage. These join based on degree of membership rather than equality, much like similarity joins. They use string distances, MinHash signatures, and learned blocking rules.

In this library, we have pushed a relational-first, similarity-as-operator approach with learned-or-derived reps as columns. This trades raw retrieval throughput (vs. dedicated vector DBs) and per-question flexibility (vs. LLM joins) for compositional algebra. The bet is that for many database-shaped workloads, composition is the missing primitive - not raw retrieval speed.

That bet extends to LLM workflows: in RAG, agent memory, and structured prompting, composition is the layer between retrieval and reasoning. We'd argue the database can do the structural work (analytics, predictions, cross-entity matches) and hand the LLM a richer conditioning set than a flat list of nearest documents.

The library was meant as a demo and just scratches the surface for what can be done. There are still many more avenues that can be explored, especially in aggregation.

## 8. Challenges

There are still major technical hurdles between where the prototype sits and similarity-join being a first-class database primitive. Below them sit per-rep concerns that persist regardless of how mature the primitive becomes.

### Structural hurdles

- **No join-time index algorithm.** Equality joins have specialised algorithms (hash-join, sort-merge) that pair with database indexes for efficient execution. Similarity joins don't have an equivalent. The vector indexes that exist (HNSW being the most common) are designed for *one-at-a-time* nearest-neighbour retrieval. Without a join-time algorithm built on them, `sim_join` falls back to computing every pairwise distance.

- **Filtered ANN.** Combining `sim_join` with a `WHERE` clause is an everyday query shape with no good answer. Pre-filtering before similarity defeats the index. Post-filtering after returns too few results. Active research area, no settled solution.

- **No cost model.** Query planners estimate equality-join result sizes from table statistics. They have no equivalent for "how many pairs within similarity ε" on a vector column. Without it, multi-step `sim_join` chains produce surprising plans, and the optimiser can't reorder joins efficiently.

- **Symmetric joins on indexes.** Vector indexes are asymmetric: one indexed side, one query side. A join is symmetric (both sides are tables). Index structures designed for symmetric similarity joins exist (M-trees, R-trees) but only as research prototypes.

- **Index update under writes.** Equality-join indexes update cheaply when rows are inserted or deleted. Vector indexes typically need rebuilding or expensive rebalancing. A production database primitive needs cheap incremental updates.

### Per-rep concerns

- **Curse of dimensionality.** As the number of dimensions grows the gap between nearest and farthest neighbour shrinks, so "similar" becomes a weaker signal.

- **The task distribution is often unknown at design time.** You'll often need to commit to a rep prior to knowing the tasks it will be used for. This is where design patterns come into play. They allow you to get more generalized representations.

- **The metric is rarely chosen, usually inherited.** Unless you create the encoding algorithm yourself, you inherit the distance metric rather than design it yourself.

- **Compositionality is fragile.** Vector arithmetic works for some tasks but can break for others.

- **Interpretability.** If the encodings are dense, they may not be easily interpretable, which makes it hard to interpret similarity.

## Closing

This is the final article introducing similarity joins. **Article 1** introduced `sim_join` as a relational operator with three cardinality shapes. **Article 2** added multiple representations per entity, M2M-derived relational reps, cross-entity joins via shared latent spaces, and group aggregation via `rep_agg`. This article named the guiding framework underneath those operators: representation from an information-theoretic view, utility as task-relative, design as pattern-heuristic bet.

The three concerns sit independently. Most confusion in representation discourse comes from conflating them:

| Concern | Kind of answer | Task-relative? | Example claim |
|---|---|---|---|
| **Definition** (what is it?) | Information-theoretic | No | "any `Z` with `I(Z; X) > 0`" |
| **Utility** (what's it for?) | Relational to a task | Yes | "`U(rep, T) = 𝔼[score(...)]`" |
| **Design** (how do you build it?) | Pattern / heuristic | Implicit task-distribution bet | "Bengio's smoothness desideratum" |

Beyond the article series, there are examples in `tutorials/examples`, as well as an overview of the api in `tutorials/api` in the Github repo. `CONTRIBUTING.md` maps what's missing in the prototype and where to plug in.

The library is a prototype, but the concepts provide a framework to approach similarity joins.

### Further reading

- Shannon 1948 (*A Mathematical Theory of Communication*) for the information-theoretic foundation.
- Fisher 1922 (*On the Mathematical Foundations of Theoretical Statistics*) for sufficient statistics.
- Tishby-Pereira-Bialek 1999 (*The Information Bottleneck Method*) for representation as compression.
- Bengio-Courville-Vincent 2013 (*Representation Learning: A Review and New Perspectives*) for the canonical ML-era pattern catalogue.
- Wolpert 1996 (*The Lack of A Priori Distinctions Between Learning Algorithms*) for the no-free-lunch theorem.
- Silva-Aref-Ali 2010 (*The Similarity Join Database Operator*) for the algebraic case for similarity as a relational operator.
- Hjaltason-Samet 1998 and Corral et al. 2000 for the spatial-database lineage of the k-closest pair shape.
- Chaudhuri-Ganti-Kaushik 2006 and Gravano et al. 2001 for the data-cleaning predecessors that motivated ε-style thresholding.

---

*The prototype used in this series is available at [github.com/short-greg/similarity-joins](https://github.com/short-greg/similarity-joins). It's a SQLAlchemy extension demonstrating similarity joins on PostgreSQL + pgvector.*